# Validate FD modes returned as TD `ModesArray`

This notebook checks the convention consistency of the LAL mode-generation paths in
`waveformtools.models.lal.LALWaveformModel`.

It is intended as a **smoke/diagnostic notebook**, not a production accuracy test.
It checks:

1. native TD mode generation returns a `waveformtools.modes_array.ModesArray`;
2. FD mode generation returns a frequency-basis `ModesArray`;
3. `get_fd_waveform_modes_as_td()` / `get_fd_modes_as_td()` returns a TD-basis `ModesArray`;
4. for an FD approximant, the automatic `get_td_waveform_modes()` route agrees with the explicit `get_fd_modes_as_td()` route;
5. the available `(ell,m)` coverage, including whether negative-`m` modes are present;
6. optional round-trip diagnostics for `FD -> TD -> FD`, useful for catching FFT/sign/warp convention issues.

The most important test for current `waveformtools` behavior is:

```python
model.get_td_waveform_modes(...) == model.get_fd_modes_as_td(...)
```

for an FD approximant such as `IMRPhenomXPHM`, because `get_td_waveform_modes()`
now delegates to the explicit FD-as-TD route for FD approximants.

In [ ]:
import numpy as np

from waveformtools.models.lal import LALWaveformModel
from waveformtools.modes_array import ModesArray

## Configuration

Adjust approximants and resolution here. Defaults are intentionally modest.

In [ ]:
# A clearly FD-domain LAL approximant.
FD_APPROXIMANT = "IMRPhenomXPHM"

# A TD-domain approximant for the native TD-mode path.
# If this is unavailable in your LALSuite build, replace it with another TD model
# supported by SimInspiralChooseTDModes in your environment.
TD_APPROXIMANT = "SEOBNRv5PHM"

# Basic BBH parameters in waveformtools naming convention.
BASE_PARAMS = dict(
    mass1=35.0,
    mass2=30.0,
    spin1x=0.0,
    spin1y=0.0,
    spin1z=0.2,
    spin2x=0.0,
    spin2y=0.0,
    spin2z=-0.1,
    distance=400.0,          # Mpc
    inclination=0.7,
    phi_ref=0.3,
    f_lower=20.0,
    f_ref=20.0,
    f_max=512.0,
    delta_t=1.0 / 4096.0,
    delta_f=1.0 / 8.0,
    ell_max=4,
)

## Helper functions

In [ ]:
def make_params(approximant):
    params = dict(BASE_PARAMS)
    params["approximant"] = approximant
    return params


def summarize_modes(wfm, name, max_print=20):
    print(f"\n{name}")
    print("-" * len(name))
    print("type:", type(wfm))
    print("label:", getattr(wfm, "label", None))
    print("ell_max:", getattr(wfm, "ell_max", None))
    print("data_len:", getattr(wfm, "data_len", None))
    print("has time_axis:", hasattr(wfm, "time_axis"))
    print("has frequency_axis:", hasattr(wfm, "frequency_axis"))
    try:
        print("time range:", float(wfm.time_axis[0]), float(wfm.time_axis[-1]), "N=", len(wfm.time_axis))
    except Exception as exc:
        print("time range: unavailable:", repr(exc))
    try:
        print("freq range:", float(wfm.frequency_axis[0]), float(wfm.frequency_axis[-1]), "N=", len(wfm.frequency_axis))
    except Exception as exc:
        print("freq range: unavailable:", repr(exc))

    present = []
    missing = []
    for ell in range(2, int(wfm.ell_max) + 1):
        for m in range(-ell, ell + 1):
            try:
                data = wfm.mode(ell, m)
                if data is None:
                    missing.append((ell, m))
                else:
                    present.append((ell, m))
            except Exception:
                missing.append((ell, m))

    print(f"present modes ({len(present)}):", present[:max_print], "..." if len(present) > max_print else "")
    print(f"missing modes ({len(missing)}):", missing[:max_print], "..." if len(missing) > max_print else "")

    neg_missing = [(ell, m) for ell, m in missing if m < 0]
    print(f"missing negative-m modes ({len(neg_missing)}):", neg_missing[:max_print], "..." if len(neg_missing) > max_print else "")
    return present, missing


def relative_l2(a, b, eps=1e-300):
    a = np.asarray(a)
    b = np.asarray(b)
    return np.linalg.norm(a - b) / max(np.linalg.norm(a), np.linalg.norm(b), eps)


def compare_common_modes(a, b, label_a="a", label_b="b", rtol=1e-10, atol=1e-12, max_print=20):
    common = []
    rows = []
    ell_max = min(int(a.ell_max), int(b.ell_max))
    for ell in range(2, ell_max + 1):
        for m in range(-ell, ell + 1):
            try:
                am = np.asarray(a.mode(ell, m))
                bm = np.asarray(b.mode(ell, m))
            except Exception:
                continue
            if am.shape != bm.shape:
                rows.append((ell, m, "shape_mismatch", am.shape, bm.shape, np.nan, np.nan))
                continue
            err_abs = np.max(np.abs(am - bm))
            err_rel = relative_l2(am, bm)
            ok = np.allclose(am, bm, rtol=rtol, atol=atol)
            rows.append((ell, m, ok, am.shape, bm.shape, float(err_abs), float(err_rel)))
            common.append((ell, m))

    failures = [r for r in rows if r[2] is not True]
    print(f"Compared {len(common)} common modes: {label_a} vs {label_b}")
    print(f"Failures / mismatches: {len(failures)}")
    for row in failures[:max_print]:
        print(row)
    return rows, failures

## 1. Native TD mode path returns `ModesArray`

In [ ]:
td_params = make_params(TD_APPROXIMANT)
td_model = LALWaveformModel(parameters_dict=td_params)

w_td_native = td_model.get_td_waveform_modes(dimensionless=False)
assert isinstance(w_td_native, ModesArray)

present_td, missing_td = summarize_modes(w_td_native, f"native TD modes: {TD_APPROXIMANT}")

## 2. FD modes and FD modes as TD return `ModesArray`

In [ ]:
fd_params = make_params(FD_APPROXIMANT)
fd_model = LALWaveformModel(parameters_dict=fd_params)

w_fd = fd_model.get_fd_waveform_modes(dimensionless=False)
w_fd_as_td = fd_model.get_fd_waveform_modes_as_td(dimensionless=False)
w_fd_as_td_alias = fd_model.get_fd_modes_as_td(dimensionless=False)

assert isinstance(w_fd, ModesArray)
assert isinstance(w_fd_as_td, ModesArray)
assert isinstance(w_fd_as_td_alias, ModesArray)

present_fd, missing_fd = summarize_modes(w_fd, f"native FD modes: {FD_APPROXIMANT}")
present_fd_td, missing_fd_td = summarize_modes(w_fd_as_td, f"FD modes as TD: {FD_APPROXIMANT}")

## 3. Explicit FD-as-TD route equals the alias

In [ ]:
rows, failures = compare_common_modes(
    w_fd_as_td,
    w_fd_as_td_alias,
    label_a="get_fd_waveform_modes_as_td",
    label_b="get_fd_modes_as_td",
    rtol=0.0,
    atol=0.0,
)
assert len(failures) == 0, "Alias and explicit FD-as-TD routes should be exactly identical."

## 4. Automatic `get_td_waveform_modes()` route for an FD approximant equals explicit FD-as-TD

In [ ]:
w_fd_auto_td = fd_model.get_td_waveform_modes(dimensionless=False)
assert isinstance(w_fd_auto_td, ModesArray)

rows, failures = compare_common_modes(
    w_fd_auto_td,
    w_fd_as_td,
    label_a="get_td_waveform_modes on FD approximant",
    label_b="get_fd_waveform_modes_as_td",
    rtol=0.0,
    atol=0.0,
)

assert len(failures) == 0, (
    "For FD approximants, get_td_waveform_modes should delegate to "
    "get_fd_waveform_modes_as_td and agree exactly."
)

## 5. Dimensionless convention check

The balance-law path uses `dimensionless=True`. This cell checks that both routes still
return `ModesArray` objects and agree exactly for the FD approximant when dimensionless
normalization is enabled.

In [ ]:
w_fd_auto_td_dimless = fd_model.get_td_waveform_modes(dimensionless=True)
w_fd_as_td_dimless = fd_model.get_fd_modes_as_td(dimensionless=True)

assert isinstance(w_fd_auto_td_dimless, ModesArray)
assert isinstance(w_fd_as_td_dimless, ModesArray)

rows, failures = compare_common_modes(
    w_fd_auto_td_dimless,
    w_fd_as_td_dimless,
    label_a="auto TD route, dimensionless",
    label_b="explicit FD-as-TD route, dimensionless",
    rtol=0.0,
    atol=0.0,
)
assert len(failures) == 0

## 6. Optional round-trip diagnostic: FD → TD → FD

This is **not** expected to be exactly zero in all cases because `to_time_basis()` calls
`undo_warp()`. It is nevertheless useful for catching gross convention problems.

Interpretation:
- small errors are reassuring;
- large errors indicate a sign/normalization/axis/warp issue worth investigating;
- failures here do not by themselves mean the explicit FD-as-TD route is inconsistent
  with the high-level `get_td_waveform_modes()` route.

In [ ]:
try:
    w_roundtrip_fd = w_fd_as_td.to_frequency_basis()
    rows, failures = compare_common_modes(
        w_roundtrip_fd,
        w_fd,
        label_a="FD -> TD -> FD",
        label_b="native FD",
        rtol=1e-6,
        atol=1e-8,
    )
    max_rel = np.nanmax([r[-1] for r in rows]) if rows else np.nan
    max_abs = np.nanmax([r[-2] for r in rows]) if rows else np.nan
    print("Round-trip max abs error:", max_abs)
    print("Round-trip max rel L2 error:", max_rel)
except Exception as exc:
    print("Round-trip diagnostic failed:", repr(exc))

## 7. Optional native TD vs FD-as-TD visual comparison

Native TD and FD approximants are generally different waveform models, so this plot is
**not** an equality test. It is only a visual sanity check of time-domain mode conventions.

The robust equality tests above are:
- explicit FD-as-TD vs alias;
- automatic FD approximant `get_td_waveform_modes()` vs explicit FD-as-TD.

In [ ]:
import matplotlib.pyplot as plt

ell, m = 2, 2

plt.figure(figsize=(7, 4))
plt.plot(w_td_native.time_axis, np.real(w_td_native.mode(ell, m)), label=f"{TD_APPROXIMANT} TD Re h{ell}{m}")
plt.plot(w_fd_as_td.time_axis, np.real(w_fd_as_td.mode(ell, m)), label=f"{FD_APPROXIMANT} FD→TD Re h{ell}{m}", alpha=0.8)
plt.xlabel("time")
plt.ylabel(f"Re h_{{{ell}{m}}}")
plt.legend()
plt.tight_layout()
plt.show()

## Expected result

The required pass/fail conditions are:

```python
isinstance(w_td_native, ModesArray)
isinstance(w_fd, ModesArray)
isinstance(w_fd_as_td, ModesArray)
get_fd_modes_as_td(...) == get_fd_waveform_modes_as_td(...)
get_td_waveform_modes(...) == get_fd_waveform_modes_as_td(...)  # for FD approximants
```

Mode coverage diagnostics should be reviewed manually. In particular, if negative-`m`
modes are missing for an approximant, `waveformtools` currently does not synthesize them
by symmetry in the `ModesArray` loader; this should be treated as an approximant/LAL
mode-content issue, not as an FD-as-TD conversion issue.